# [감사 · 2026-10-25] 평가 시작 상태가 성과에 주는 영향

원본 평가(`run_beta_smoothed`)는 평가 구간 첫날 **44종목 등가중 100%** 에서 출발해 α=0.01 스무딩으로 천천히 정책 비중으로 옮겨 간다.
검증(2025-10-01)·테스트(2026-01-01) 모두 시작 직후 3일간 등가중이 +9.6%·+10.9% 올라, 이 초기 노출만으로 약 +6%p가 생긴 것으로
추정됐다(로컬 근사). 이 노트북은 시작 상태만 바꿔 4개 β를 다시 평가해 그 영향을 확정한다. **학습 없음.**

| 시작 방식 | 내용 |
|---|---|
| orig | 원본 그대로 — 첫날 등가중 100%(논문 수치 재현용) |
| cash | 첫날 전액 현금(실계좌와 같은 조건) |
| warmup | 평가 시작 14일 전부터 현금 상태로 정책을 미리 돌리고, 수익은 평가 시작일부터만 잰다(시작 상태 영향 제거 — 권장 기준) |

비교 기준: 비트코인 보유, 44종목 등가중(매 스텝 재조정), 등가중 × β=30 노출 경로(비용 미포함).

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 충분, 약 30~60분.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/crypto_start_state_audit.csv`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·평가 함수 (β=30으로 치환) — 원본 crypto_beta_robustness_colab(커밋 0e86a41) 셀 재사용
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


## 1. 4개 β × 시작 방식 3가지 + 비교 기준

In [ ]:
WARMUP = 14 * 48      # 14일 × 하루 48버킷 — α=0.01이면 0.99^672 ≈ 0.001로 시작 상태 영향이 사라진다
BETAS = [-90, -30, 30, 90]
POOL_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
models = {b: PPO.load(str(POOL_V4 / f'agent_beta_{b}.zip')) for b in BETAS}
print('로드:', {b: str(POOL_V4 / f'agent_beta_{b}.zip') for b in BETAS})


def run_policy(model, buckets, start='orig', alpha=ALPHA, top_k=TOP_K):
    """run_beta_smoothed와 같은 루프에 시작 상태만 바꾼다. 반환: (평가 구간 가치 곡선, 턴오버, 스텝별 노출)."""
    if start == 'warmup':
        i0 = all_buckets.index(buckets[0])
        lo = max(0, i0 - WARMUP)
        run_bk, skip = all_buckets[lo:i0] + list(buckets), i0 - lo
    else:
        run_bk, skip = list(buckets), 0
    env = SparseGatedCryptoPortfolioEnv(buckets=run_bk, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    if start in ('cash', 'warmup'):
        env.weights = np.zeros(env.n_assets, dtype=np.float32)
        obs = env._get_obs()
    expo, done = [], False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        tw, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * tw
        expo.append(float(w.sum()))
        obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    vals = np.asarray(env.value_history[skip:], dtype=np.float64)
    return vals / vals[0], list(env.turnover_history[skip:]), np.asarray(expo[skip:])


def ew_returns(buckets):
    """매 버킷 거래 가능 종목 등가중 수익률."""
    out = []
    for b in buckets[:-1]:
        r = [ret_index[(s, b)] for s in symbols if (s, b) in ret_index]
        out.append(np.mean(r) if r else 0.0)
    return np.asarray(out)


assert 'BTCUSDT' in symbols, symbols[:5]
rows, curves = [], {}
for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
    r_ew = ew_returns(bk)
    btc = np.asarray([ret_index.get(('BTCUSDT', b), 0.0) for b in bk[:-1]])
    for name, r in [('BTC 보유', btc), ('44종목 등가중(무비용 근사)', r_ew)]:
        v = np.concatenate([[1.0], np.cumprod(1 + r)])
        rows.append({'구간': split, '전략': name, '시작': '-', **compute_perf(v), '평균 노출': 1.0})
        curves[(split, name, '-')] = v
    ew_vals, ew_turns = equal_weight_backtest(CryptoPortfolioEnv(buckets=bk, **env_kwargs))
    rows.append({'구간': split, '전략': '44종목 등가중(원본, 비용 포함)', '시작': '-', **perf_row(ew_vals, ew_turns), '평균 노출': 1.0})
    for beta, model in models.items():
        for start in ['orig', 'cash', 'warmup']:
            print(f'{split} β={beta} {start} 실행 중...')
            vals, turns, expo = run_policy(model, bk, start)
            rows.append({'구간': split, '전략': f'β={beta}', '시작': start, **perf_row(vals, turns), '평균 노출': expo.mean()})
            curves[(split, f'β={beta}', start)] = vals
            if beta == 30:  # 노출 맞춤 벤치마크: 등가중을 정책의 노출 경로만큼만 보유(비용 미포함)
                v = np.concatenate([[1.0], np.cumprod(1 + expo[:len(r_ew)] * r_ew[:len(expo)])])
                rows.append({'구간': split, '전략': '등가중×β=30 노출 경로', '시작': start, **compute_perf(v), '평균 노출': expo.mean()})

audit = pd.DataFrame(rows)
audit.to_csv(CRYPTO_DIR / 'crypto_start_state_audit.csv', index=False, encoding='utf-8-sig')
show = audit[['구간', '전략', '시작', '총 수익률', 'Sharpe', 'MDD', '평균 노출']].copy()
for k, f in {'총 수익률': '{:+.2%}', 'Sharpe': '{:.3f}', 'MDD': '{:.2%}', '평균 노출': '{:.1%}'}.items():
    show[k] = show[k].map(f.format)
pd.set_option('display.width', 200)
print(show.to_string(index=False))
print('\n저장 완료: crypto_start_state_audit.csv')


## 2. 재현 확인과 요약

In [ ]:
# 재현 확인 + 요약: orig가 논문 수치(β=30 valid +2.98%·test +8.73%)와 같은지, 시작 상태를 바꾸면 얼마나 변하는지
paper = {('valid', 'β=30'): 0.0298, ('test', 'β=30'): 0.0873}
for (split, strat), p in paper.items():
    got = audit[(audit['구간'] == split) & (audit['전략'] == strat) & (audit['시작'] == 'orig')]['총 수익률'].iloc[0]
    print(f'[재현] {split} {strat} orig {got:+.2%} (논문 {p:+.2%}) {"OK" if abs(got - p) < 0.002 else "← 불일치, 확인 필요"}')
print()
for split in ['valid', 'test']:
    print(f'== {split}')
    for beta in BETAS:
        r = {s: audit[(audit['구간'] == split) & (audit['전략'] == f'β={beta}') & (audit['시작'] == s)].iloc[0] for s in ['orig', 'cash', 'warmup']}
        print(f'  β={beta:>3}: orig {r["orig"]["총 수익률"]:+.2%} (샤프 {r["orig"]["Sharpe"]:.2f}) → '
              f'warmup {r["warmup"]["총 수익률"]:+.2%} (샤프 {r["warmup"]["Sharpe"]:.2f}) | cash {r["cash"]["총 수익률"]:+.2%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, split in zip(axes, ['valid', 'test']):
    ax.plot(curves[(split, 'BTC 보유', '-')], color='orange', label='BTC buy&hold')
    for start, ls in [('orig', '--'), ('warmup', '-')]:
        ax.plot(curves[(split, 'β=30', start)], ls=ls, color='tab:blue', label=f'beta=30 ({start})')
    ax.set_title(split); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(CRYPTO_DIR / 'crypto_start_state_audit.png', dpi=120); plt.show()
